In [1]:
import decimal
import os

import numpy as np
import pandas as pd

In [2]:
MODEL_NAME = 'ChIEDiff'

# Load Data

In [3]:
class Globals:
    root = '.' if os.path.isdir('results') else '..'
    molecules_path = 'archive/chiediff_generated_molecules.pt'

    main_table = 'results/comparison/f1_sbdd/main_table/main_table.csv'
    redocking = 'results/comparison/f2_pose_fidelity/redocking_consistency/redocking_consistency.csv'
    nci = 'results/comparison/f3_nci/nci_summary_strict.csv'
    pb_distance = 'results/comparison/f2_pose_fidelity/pb_distance/pb_distance_strict.csv'
    pose = 'results/comparison/f2_pose_fidelity/pose_stability/pose_stability_strict.csv'
    heavy_clash = 'results/pose_fidelity/heavy_clash/table_inputs.csv'
    properties = 'results/comparison/f1_sbdd/properties/property_overall.csv'
    bond_jsd = 'results/comparison/f1_sbdd/bond_jsd/bond_jsd_overall.csv'
    ring_size = 'results/comparison/f1_sbdd/ring_size/ring_size_overall.csv'
    posebusters = 'results/comparison/appendix/posebusters/pose_quality_summary_posebusters.csv'
    specificity = 'results/comparison/master_table.csv'
    delta_per_mol = 'results/comparison/f4_delta_score/delta_score_per_mol.csv.gz'
    ablation = 'results/diagnostics/ablation_nci_v2/posecheck_ablation_all.csv'
    dock_success = 'results/diagnostics/ablation_nci_v2/dock_success.csv'
    plip_intersection = 'results/diagnostics/ablation_nci_v2/plip_intersection.csv'

In [4]:
def load(path, index='id'):
    df = pd.read_csv(os.path.join(Globals.root, path))
    return df.set_index(index) if index else df

main = load(Globals.main_table)
redock = load(Globals.redocking)
nci = load(Globals.nci)
pbd = load(Globals.pb_distance)
pose = load(Globals.pose)
heavy = load(Globals.heavy_clash, index=None)
prop = load(Globals.properties)
bond = load(Globals.bond_jsd)
ring = load(Globals.ring_size)
pb = load(Globals.posebusters, index=None)
master = load(Globals.specificity)
delta = load(Globals.delta_per_mol, index=None)
abl = load(Globals.ablation)
dock_success = load(Globals.dock_success)
plip_x = load(Globals.plip_intersection)

In [5]:
# Rows in the paper's order. ARMS: (energy channels, guidance off, guidance on).
MODELS = {'reference': 'Test set', 'pocket2mol': 'Pocket2Mol', 'targetdiff': 'TargetDiff',
          'ipdiff': 'IPDiff', 'alidiff': 'AliDiff', 'molcraft': 'MolCRAFT',
          'pidiff_retrain': 'PIDiff', 'kgdiff': 'KGDiff', 'ours_vina': MODEL_NAME}
ARMS = [('No energy', 'abl_none_off', 'novdw'),
        ('Ste', 'abl_vdw_off', 'abl_vdw_on'),
        ('Hb', 'abl_hbond_off', 'abl_hbond_on'),
        ('Hp', 'abl_hydro_off', 'abl_hydro_on'),
        ('Ste+Hb', 'abl_vdw_hbond_off', 'abl_vdw_hbond_on'),
        ('Ste+Hp', 'abl_vdw_hydro_off', 'abl_vdw_hydro_on'),
        ('Hb+Hp', 'abl_hbond_hydro_off', 'abl_hbond_hydro_on'),
        ('Ste+Hb+Hp', 'ours_noguide', 'ours_vina')]
RULES = (0, 7)


def fmt(value, digits):
    # round half up, as the paper's tables are typeset
    if isinstance(value, str):
        return value
    if value is None or pd.isna(value):
        return '--'
    step = decimal.Decimal(1).scaleb(-digits)
    return str(decimal.Decimal(repr(float(value))).quantize(step, decimal.ROUND_HALF_UP))


def rows_of(df, ids, fields):
    return [[MODELS[i]] + [df.at[i, f] for f in fields] for i in ids]


def spans(heads, level):
    j = 0
    while j < len(heads):
        k = j
        while k + 1 < len(heads) and heads[k + 1][:level + 1] == heads[j][:level + 1]:
            k += 1
        yield j, k
        j = k + 1


def print_table(title, columns, rows, rules=()):
    # columns: (header tuple, digits); digits None for a text column
    heads = [h for h, _ in columns]
    body = [[v if d is None else fmt(v, d) for v, (_, d) in zip(r, columns)] for r in rows]
    width = [max([len(h[-1])] + [len(r[j]) for r in body]) for j, h in enumerate(heads)]
    for level in range(len(heads[0]) - 1):
        for j, k in spans(heads, level):
            short = max(len(heads[j][level]) - sum(width[j:k + 1]) - 2 * (k - j), 0)
            for c in range(j, k + 1):
                width[c] += short // (k - j + 1) + (c == k) * (short % (k - j + 1))
    align = [str.ljust if d is None else str.rjust for _, d in columns]
    line = lambda cells: '  '.join(cells).rstrip()
    head = [line(heads[j][level].center(sum(width[j:k + 1]) + 2 * (k - j))
                 for j, k in spans(heads, level)) for level in range(len(heads[0]) - 1)]
    head.append(line(a(h[-1], w) for a, h, w in zip(align, heads, width)))
    out = [line(a(c, w) for a, c, w in zip(align, r, width)) for r in body]
    rule = '-' * max(len(x) for x in head + out)
    text = [title, '=' * len(title)] + head + [rule]
    for i, r in enumerate(out):
        text += [r, rule] if i in rules else [r]
    print('\n'.join(text) + '\n')

# Main Tables

## Docking Scores

In [6]:
columns, fields = [(('', '', 'Model'), None)], []
for name, key, unit in [('Vina', 'vina', 'kcal/mol'), ('Vinardo', 'vinardo', 'kcal/mol'),
                        ('CNNaff', 'cnnaff', 'pK')]:
    for state, part in [('Gen', 'score'), ('Dock', 'dock')]:
        for stat in ['Avg', 'Med']:
            columns.append(((f'{name} ({unit})', state, stat), 2))
            fields.append(f'{key}_{part}_{stat.lower()}')
print_table('Generated-pose and redocked scoring statistics', columns,
            rows_of(main, MODELS, fields), RULES)

Generated-pose and redocked scoring statistics
                  Vina (kcal/mol)             Vinardo (kcal/mol)           CNNaff (pK)
                 Gen            Dock          Gen           Dock         Gen         Dock
Model         Avg     Med     Avg    Med    Avg    Med    Avg    Med   Avg   Med   Avg   Med
--------------------------------------------------------------------------------------------
Test set    -6.36   -6.46   -7.45  -7.26  -6.32  -6.31  -7.75  -7.36  5.55  5.61  5.61  5.65
--------------------------------------------------------------------------------------------
Pocket2Mol  -5.14   -4.70   -7.15  -6.79  -4.19  -4.01  -6.89  -6.46  4.89  4.74  5.09  4.92
TargetDiff  -5.47   -6.30   -7.80  -7.91  -4.42  -5.19  -7.26  -7.38  5.73  5.88  5.78  5.92
IPDiff      -6.41   -7.00   -8.57  -8.51  -4.99  -5.69  -7.65  -7.71  5.84  6.04  5.89  6.10
AliDiff     -6.97   -7.74   -8.65  -8.64  -6.10  -6.74  -7.77  -7.85  5.98  6.21  6.00  6.20
MolCRAFT    -6.59   -7.04   -7.9

## Redocking Consistency

In [7]:
columns, fields = [(('', '', 'Model'), None)], []
for name, key in [('Vina', 'vina'), ('Vinardo', 'vinardo'), ('GNINA', 'gnina')]:
    for head, part in [('Gap', 'gap'), ('RMSD', 'rmsd')]:
        for stat in ['Avg', 'Med']:
            columns.append(((name, head, stat), 2))
            fields.append(f'{key}_{part}_{stat.lower()}')
    columns.append(((name, '<2 A', '%'), 1))
    fields.append(f'{key}_rmsd_pct_lt2')
print_table('Redocking score gaps and pose displacement', columns,
            rows_of(redock, MODELS, fields), RULES)

Redocking score gaps and pose displacement
                        Vina                        Vinardo                        GNINA
               Gap         RMSD     <2 A     Gap         RMSD     <2 A     Gap         RMSD     <2 A
Model        Avg   Med   Avg   Med     %   Avg   Med   Avg   Med     %   Avg   Med   Avg   Med     %
----------------------------------------------------------------------------------------------------
Test set    1.31  0.70  5.15  5.63  24.5  1.48  1.08  4.05  3.82  29.6  0.78  0.27  2.66  1.80  52.0
----------------------------------------------------------------------------------------------------
Pocket2Mol  2.38  1.93  4.74  4.34  26.8  2.71  2.30  3.95  3.75  31.5  2.14  1.46  3.68  3.21  34.1
TargetDiff  2.20  1.19  4.58  4.65  28.8  2.79  1.94  3.81  3.56  36.2  2.67  1.10  3.82  3.43  33.8
IPDiff      2.01  1.04  4.59  4.72  29.2  2.80  1.69  3.67  3.28  39.9  3.63  1.17  4.41  4.41  29.6
AliDiff     1.85  0.67  4.15  3.92  36.9  2.07  1.11  3.19  

## Interactions and Clashes

In [8]:
PROLIF = [('Acc.', 'prolif_HBAcceptor'), ('Don.', 'prolif_HBDonor'), ('Hyd.', 'prolif_Hydrophobic'),
          ('VdW', 'prolif_VdWContact'), ('Total', 'prolif_total'), ('Total/HA', 'prolif_per_heavy')]
quality = nci.join(pbd['pb_clashes_protein_avg']).join(
    heavy[heavy.table == 'tab:quality'].set_index('id')['heavy_clash_avg'])

columns = [(('', 'Model'), None)] + [(('ProLIF interactions', h), 2) for h, _ in PROLIF] + \
          [(('Clash', 'PoseBusters'), 2), (('Clash', 'PoseCheck'), 2)]
fields = [f for _, f in PROLIF] + ['pb_clashes_protein_avg', 'heavy_clash_avg']
print_table('Intermolecular interactions and physical clashes of generated poses', columns,
            rows_of(quality, MODELS, fields), RULES)

Intermolecular interactions and physical clashes of generated poses
                      ProLIF interactions                     Clash
Model       Acc.  Don.  Hyd.    VdW  Total  Total/HA  PoseBusters  PoseCheck
----------------------------------------------------------------------------
Test set    2.07  0.51  0.73   8.87  12.18      0.58         0.05       0.52
----------------------------------------------------------------------------
Pocket2Mol  0.99  0.36  0.89   7.03   9.27      0.58         0.25       1.15
TargetDiff  1.71  0.73  1.22  10.26  13.91      0.61         0.38       1.71
IPDiff      1.19  0.46  1.51   9.76  12.92      0.57         0.35       1.57
AliDiff     1.19  0.44  1.80  10.01  13.45      0.58         0.23       1.14
MolCRAFT    1.92  0.78  0.99   9.56  13.25      0.64         0.21       1.01
PIDiff      1.13  0.35  1.07   7.97  10.52      0.47         0.19       0.77
KGDiff      1.60  0.64  1.47   9.85  13.56      0.59         0.20       0.84
-----------------

## Ablation

In [9]:
ABL = [('Acc.', 'int_HBAcceptor_mean', 2), ('Don.', 'int_HBDonor_mean', 2),
       ('Hyd.', 'int_Hydrophobic_mean', 2), ('VdW', 'int_VdWContact_mean', 2),
       ('Total', 'inter_mean', 2), ('Total/HA', 'inter_per_atom', 3)]
ablation = abl.join(heavy[heavy.table == 'tab:ablation-main'].set_index('id')['heavy_clash_avg'])

columns = [(('', 'Energy'), None), (('', 'Guide'), None)] + \
          [(('ProLIF interactions', h), d) for h, _, d in ABL] + \
          [(('PoseCheck', 'Clash'), 2), (('PoseCheck', 'Strain'), 1)]
fields = [f for _, f, _ in ABL] + ['heavy_clash_avg', 'strain_median']
rows = []
for guide, side in [('OFF', 1), ('ON', 2)]:
    rows += [[arm[0], guide] + [ablation.at[arm[side], f] for f in fields]
             for arm in ARMS if arm[0] in ('No energy', 'Ste', 'Ste+Hb', 'Ste+Hp', 'Ste+Hb+Hp')]
print_table('Contributions of energy learning and affinity guidance', columns, rows, rules=(4,))

Contributions of energy learning and affinity guidance
                            ProLIF interactions               PoseCheck
Energy     Guide  Acc.  Don.  Hyd.    VdW  Total  Total/HA  Clash  Strain
-------------------------------------------------------------------------
No energy  OFF    1.59  0.75  1.23   9.87  13.44     0.604   1.79   257.1
Ste        OFF    1.59  0.39  1.21   8.77  11.96     0.537   0.64   700.5
Ste+Hb     OFF    2.08  0.87  1.28   9.81  14.03     0.630   0.56  1031.0
Ste+Hp     OFF    1.50  0.38  1.37   8.92  12.18     0.552   0.63   637.6
Ste+Hb+Hp  OFF    2.01  0.88  1.43  10.07  14.39     0.645   0.61   624.1
-------------------------------------------------------------------------
No energy  ON     1.61  0.76  1.36   9.64  13.37     0.593   0.89   440.2
Ste        ON     1.65  0.39  1.60   8.99  12.62     0.565   1.05   923.3
Ste+Hb     ON     2.08  0.83  1.53  10.11  14.55     0.638   0.58  1230.6
Ste+Hp     ON     1.22  0.33  1.61   8.50  11.66     0.531 

# Supplementary Tables

## Cohorts

In [10]:
# the per-molecule specificity file names ChIEDiff by its run tag
DELTA_ID = {i: ('vina_fixed_on' if i == 'ours_vina' else i) for i in MODELS}
pairs = delta[np.isfinite(delta.on_dock) & np.isfinite(delta.off_dock)]
COMMON = set.intersection(*(set(pairs.loc[pairs.model == m, 'pocket_idx']) for m in DELTA_ID.values()))

cohort = pd.DataFrame({
    'score_n': main['n_mols'], 'nci_n': nci['n'], 'pose_n': pose['n_mols'],
    'chem_n': prop['n_mols'] - prop['n_unmeasurable'],
    'spec_n': pd.Series({i: ((pairs.model == m) & pairs.pocket_idx.isin(COMMON)).sum()
                         for i, m in DELTA_ID.items()}),
    'pockets': main['n_pockets']})
columns = [(('Model',), None)] + [((h,), 0) for h in
           ['Docking Score', 'NCI', 'Physical checks', 'Chemistry', 'Specificity', 'Pockets']]
print_table('Measured molecules by cohort', columns, rows_of(cohort, MODELS, cohort.columns), RULES)

Measured molecules by cohort
Model       Docking Score   NCI  Physical checks  Chemistry  Specificity  Pockets
---------------------------------------------------------------------------------
Test set              100   100              100        100           99      100
---------------------------------------------------------------------------------
Pocket2Mol           9831  9802             9801       9831         9731      100
TargetDiff           9036  8954             8954       9036         8992      100
IPDiff               9011  8705             8689       9007         8994      100
AliDiff              9236  8864             8858       9232         9188      100
MolCRAFT             9667  9657             9657       9667         9573      100
PIDiff               8654  8476             8469       8654         8626      100
KGDiff               8813  8280             8278       8813         8808      100
---------------------------------------------------------------------

## Interaction Profiles

In [11]:
PLIP = [('H-bond', 'plip_n_hbond'), ('Hyd.', 'plip_n_hydrophobic'), ('Total', 'plip_total'),
        ('Total/HA', 'plip_per_heavy')]
columns = [(('', 'Model'), None), (('', 'Heavy'), 1)] + \
          [(('PLIP', h), 2) for h, _ in PLIP] + [(('ProLIF', h), 2) for h, _ in PROLIF]
fields = ['heavy_mean'] + [f for _, f in PLIP] + [f for _, f in PROLIF]
print_table('Interaction profiles measured on generated poses', columns,
            rows_of(nci, MODELS, fields), RULES)

Interaction profiles measured on generated poses
                                PLIP                               ProLIF
Model       Heavy  H-bond  Hyd.  Total  Total/HA  Acc.  Don.  Hyd.    VdW  Total  Total/HA
------------------------------------------------------------------------------------------
Test set     22.8    3.92  3.60   8.45      0.39  2.07  0.51  0.73   8.87  12.18      0.58
------------------------------------------------------------------------------------------
Pocket2Mol   17.7    2.37  3.51   6.59      0.40  0.99  0.36  0.89   7.03   9.27      0.58
TargetDiff   24.1    4.09  4.80   9.64      0.42  1.71  0.73  1.22  10.26  13.91      0.61
IPDiff       23.8    2.74  6.21   9.51      0.41  1.19  0.46  1.51   9.76  12.92      0.57
AliDiff      24.1    2.67  6.91   9.91      0.43  1.19  0.44  1.80  10.01  13.45      0.58
MolCRAFT     22.7    4.04  4.48   9.61      0.45  1.92  0.78  0.99   9.56  13.25      0.64
PIDiff       23.9    3.32  4.25   8.23      0.36  1.13  0.

## Ablation Lattice

In [12]:
lattice = ablation.join(dock_success['dock_success_pct']).join(
    plip_x[['hbond', 'hydrophobic', 'per_heavy']])

columns = [(('', 'Energy'), None), (('', 'Yield %'), 1),
           (('PLIP', 'H-bond'), 2), (('PLIP', 'Hyd.'), 2), (('PLIP', 'Total/HA'), 3)] + \
          [(('ProLIF', h), d) for h, f, d in ABL if f != 'inter_mean'] + \
          [(('PoseCheck', 'Clash'), 2), (('PoseCheck', 'Strain'), 1)]
fields = ['dock_success_pct', 'hbond', 'hydrophobic', 'per_heavy'] + \
         [f for _, f, _ in ABL if f != 'inter_mean'] + ['heavy_clash_avg', 'strain_median']
rows = []
for guide, side in [('OFF', 1), ('ON', 2)]:
    rows.append([f'Guidance {guide}'] + [''] * len(fields))
    rows += [[arm[0]] + [lattice.at[arm[side], f] for f in fields] for arm in ARMS]
print_table('Energy-term and guidance lattice', columns, rows, rules=(8,))

Energy-term and guidance lattice
                                PLIP                         ProLIF                 PoseCheck
Energy        Yield %  H-bond  Hyd.  Total/HA  Acc.  Don.  Hyd.    VdW  Total/HA  Clash  Strain
-----------------------------------------------------------------------------------------------
Guidance OFF
No energy        87.4    3.74  4.35     0.391  1.59  0.75  1.23   9.87     0.604   1.79   257.1
Ste              90.0    3.70  5.41     0.426  1.59  0.39  1.21   8.77     0.537   0.64   700.5
Hb               58.4    3.44  4.70     0.429  1.47  1.04  1.18   9.84     0.690   4.24   221.7
Hp               93.6    4.01  5.02     0.426  1.67  0.79  1.36  10.51     0.636   2.56   305.2
Ste+Hb           86.9    4.22  5.42     0.453  2.08  0.87  1.28   9.81     0.630   0.56  1031.0
Ste+Hp           86.3    3.58  5.74     0.438  1.50  0.38  1.37   8.92     0.552   0.63   637.6
Hb+Hp            59.7    3.45  5.07     0.440  1.53  1.12  1.32   9.99     0.700   4.79   25

## Properties

In [13]:
props = prop.join(bond[['ejsd', 'jsd_atom']]).join(
    ring[['mean_rings_per_mol', 'acyclic_pct', 'ring_ge10_share_pct']])
PROPS = [('Molecular properties', 'QED', 'qed_avg', 3),
         ('Molecular properties', 'SA', 'sa_avg', 3),
         ('Molecular properties', 'logP', 'logp_avg', 2),
         ('Molecular properties', 'TPSA (A^2)', 'tpsa_avg', 1),
         ('Molecular properties', 'MW (Da)', 'mw_avg', 1),
         ('Molecular properties', '5-rule %', 'lipinski_pass5_pct', 1),
         ('Structural distributions', 'Bond JS', 'ejsd', 4),
         ('Structural distributions', 'Atom JS', 'jsd_atom', 4),
         ('Structural distributions', 'Rings/mol', 'mean_rings_per_mol', 3),
         ('Structural distributions', 'Acyclic %', 'acyclic_pct', 1),
         ('Structural distributions', 'Ring >=10 %', 'ring_ge10_share_pct', 1)]
columns = [(('', 'Model'), None)] + [((g, h), d) for g, h, _, d in PROPS]
print_table('Molecular properties and structural distribution summaries', columns,
            rows_of(props, MODELS, [f for _, _, f, _ in PROPS]), RULES)

Molecular properties and structural distribution summaries
                           Molecular properties                              Structural distributions
Model         QED     SA  logP  TPSA (A^2)  MW (Da)  5-rule %  Bond JS  Atom JS  Rings/mol  Acyclic %  Ring >=10 %
------------------------------------------------------------------------------------------------------------------
Test set    0.476  0.728  0.89       117.1    332.6      65.0       --   0.0694      2.470       13.0          2.0
------------------------------------------------------------------------------------------------------------------
Pocket2Mol  0.573  0.756  1.51        63.5    241.4      89.9   0.4612   0.0923      2.845        5.5          0.7
TargetDiff  0.480  0.585  1.36       100.1    344.1      70.3   0.3778   0.0596      2.868        7.1          3.5
IPDiff      0.518  0.593  3.58        59.6    331.9      63.2   0.4298   0.1500      3.366        2.8          4.3
AliDiff     0.503  0.565  4.38    

## Bond Length JSD

In [14]:
BONDS = ['C-C', 'C=C', 'C:C', 'C-N', 'C=N', 'C:N', 'C-O', 'C=O', 'C:O']
columns = [(('Model',), None)] + [((b,), 2) for b in BONDS]
print_table('Bond-type distribution distances', columns,
            rows_of(bond, MODELS, [f'jsd_{b}' for b in BONDS]), RULES)

Bond-type distribution distances
Model        C-C   C=C   C:C   C-N   C=N   C:N   C-O   C=O   C:O
----------------------------------------------------------------
Test set      --    --    --    --    --    --    --    --    --
----------------------------------------------------------------
Pocket2Mol  0.50  0.56  0.42  0.43  0.63  0.49  0.45  0.52  0.78
TargetDiff  0.37  0.50  0.26  0.36  0.55  0.24  0.42  0.46  0.80
IPDiff      0.45  0.53  0.22  0.41  0.57  0.25  0.49  0.43  0.83
AliDiff     0.39  0.57  0.33  0.42  0.62  0.39  0.50  0.44  0.79
MolCRAFT    0.32  0.50  0.32  0.30  0.52  0.25  0.35  0.33  0.81
PIDiff      0.53  0.56  0.37  0.44  0.63  0.40  0.53  0.46  0.79
KGDiff      0.38  0.56  0.20  0.38  0.58  0.33  0.44  0.43  0.76
----------------------------------------------------------------
ChIEDiff    0.49  0.60  0.40  0.46  0.65  0.46  0.52  0.56  0.83



## PoseBusters

In [15]:
pb_ds = pb[pb.subset == 'docking_successful']
pb_ds = pb_ds.assign(id=pb_ds.model.map({label: i for i, label in main['label'].items()}))
pb_ds = pb_ds.dropna(subset=['id']).set_index('id').join(pose['strain_med'])

CHECKS = [('n', 'n_mol', 0), ('All %', 'pass_all_pct', 2), ('Lengths %', 'bond_lengths', 2),
          ('Angles %', 'bond_angles', 2), ('PL clash %', 'minimum_distance_to_protein', 2),
          ('Int. clash %', 'internal_steric_clash', 2), ('Energy %', 'internal_energy', 2),
          ('Strain', 'strain_med', 1)]
columns = [(('Model',), None)] + [((h,), d) for h, _, d in CHECKS]
print_table('PoseBusters pass rates and PoseCheck strain', columns,
            rows_of(pb_ds, [i for i in MODELS if i != 'reference'], [f for _, f, _ in CHECKS]),
            rules=(6,))

PoseBusters pass rates and PoseCheck strain
Model          n  All %  Lengths %  Angles %  PL clash %  Int. clash %  Energy %  Strain
----------------------------------------------------------------------------------------
Pocket2Mol  9831  72.34      91.84     98.93       82.51         99.20     98.47     4.0
TargetDiff  9025  51.65      98.85     73.93       87.36         91.15     97.04   179.5
IPDiff      8965  37.46      95.92     55.15       87.96         88.85     92.46   367.0
AliDiff     9180  32.80      94.66     44.01       93.22         85.50     93.06  1832.0
MolCRAFT    9665  76.72      99.84     97.14       96.18         96.52     97.66    18.5
PIDiff      8559  35.92      90.92     51.71       97.89         77.40     90.50   564.9
KGDiff      8498  28.98      88.48     37.26       97.48         81.22     87.01   643.5
----------------------------------------------------------------------------------------
ChIEDiff    8271  35.15      63.57     56.67       98.15         8

## Pocket Specificity

In [16]:
SPEC = [(('Delta (kcal/mol)', 'Avg'), 'f4_delta_p_avg', 2), (('Delta (kcal/mol)', 'Med'), 'f4_delta_p_med', 2),
        (('', 'Ratio %'), 'f4_ratio_pct_avg', 1), (('', 'Win %'), 'f4_win_pct_avg', 1),
        (('Vina (kcal/mol)', 'On-Tgt'), 'f4_on_med', 2), (('Vina (kcal/mol)', 'Off-Tgt'), 'f4_off_med', 2)]
columns = [(('', 'Model'), None)] + [(h, d) for h, _, d in SPEC]
print_table('Exploratory pocket specificity under a fixed random off-target assignment', columns,
            rows_of(master, MODELS, [f for _, f, _ in SPEC]), RULES)

Exploratory pocket specificity under a fixed random off-target assignment
            Delta (kcal/mol)                  Vina (kcal/mol)
Model           Avg      Med  Ratio %  Win %  On-Tgt  Off-Tgt
-------------------------------------------------------------
Test set       0.78     0.24       --   58.6   -7.20    -6.96
-------------------------------------------------------------
Pocket2Mol     0.24     0.19     30.7   50.4   -6.94    -6.92
TargetDiff     0.62     0.43     38.6   56.7   -7.99    -7.42
IPDiff         0.83     0.31     43.3   56.3   -8.69    -8.07
AliDiff        1.26     0.86     49.1   62.0   -8.82    -7.81
MolCRAFT       0.63     0.40     39.4   58.3   -7.92    -7.38
PIDiff         0.85     0.56     44.4   59.6   -8.25    -7.72
KGDiff         1.52     1.03     56.4   64.5   -9.26    -8.13
-------------------------------------------------------------
ChIEDiff       2.03     1.76     65.6   71.9   -9.94    -7.97



### Re-aggregated from the per-molecule file

In [17]:
# clip to [-20, 20] against out-of-box penalties, average within each pocket, then across pockets
d = delta.dropna(subset=['on_dock', 'off_dock'])
d = d[d.pocket_idx.isin(COMMON)].assign(on=lambda x: x.on_dock.clip(-20, 20),
                                        off=lambda x: x.off_dock.clip(-20, 20))
d = d.assign(delta=d.off - d.on)
per_pocket = d.groupby(['model', 'pocket_idx'])[['delta', 'on', 'off']].mean()
recomputed = per_pocket.groupby('model').agg(
    f4_delta_p_avg=('delta', 'mean'), f4_delta_p_med=('delta', 'median'),
    f4_on_med=('on', 'median'), f4_off_med=('off', 'median'))
recomputed = recomputed.rename(index={m: i for i, m in DELTA_ID.items()})

KEEP = [s for s in SPEC if s[1] in recomputed.columns]
print_table(f'Re-aggregated, {len(COMMON)} common pockets', [(('', 'Model'), None)] + [(h, d) for h, _, d in KEEP],
            rows_of(recomputed, MODELS, [f for _, f, _ in KEEP]), RULES)
differ = [(MODELS[i], f) for i in MODELS for _, f, dg in KEEP
          if fmt(recomputed.at[i, f], dg) != fmt(master.at[i, f], dg)]
print('%d cells compared, %d differ' % (len(MODELS) * len(KEEP), len(differ)), *differ)

Re-aggregated, 99 common pockets
            Delta (kcal/mol)  Vina (kcal/mol)
Model           Avg      Med  On-Tgt  Off-Tgt
---------------------------------------------
Test set       0.78     0.24   -7.20    -6.96
---------------------------------------------
Pocket2Mol     0.24     0.19   -6.94    -6.92
TargetDiff     0.62     0.43   -7.99    -7.42
IPDiff         0.83     0.31   -8.69    -8.07
AliDiff        1.26     0.86   -8.82    -7.81
MolCRAFT       0.63     0.40   -7.92    -7.38
PIDiff         0.85     0.56   -8.25    -7.72
KGDiff         1.52     1.03   -9.26    -8.13
---------------------------------------------
ChIEDiff       2.03     1.76   -9.94    -7.97

36 cells compared, 0 differ


# Metrics Summary

In [18]:
if os.path.isfile(os.path.join(Globals.root, Globals.molecules_path)):
    import torch
    from rdkit import RDLogger
    RDLogger.DisableLog('rdApp.*')
    path = os.path.join(Globals.root, Globals.molecules_path)
    try:                                  # torch < 1.13 has no weights_only
        Globals.our_results = torch.load(path, map_location='cpu', weights_only=False)
    except TypeError:
        Globals.our_results = torch.load(path, map_location='cpu')
else:
    Globals.our_results = None
    print('%s not found; download it with\n'
          '    python tools/download_archive.py --doi 10.5281/zenodo.23029438' % Globals.molecules_path)

In [19]:
def docked(r):
    dock = (r.get('vina') or {}).get('dock')
    return bool(dock) and dock[0].get('affinity') is not None


def print_results(results):
    chem = [r['chem_results'] for r in results if r.get('chem_results')]
    rings = [{int(k): v for k, v in c['ring_size'].items()} for c in chem]
    n_rings = np.array([sum(rs.values()) for rs in rings])
    n_large = np.array([sum(v for k, v in rs.items() if k >= 10) for rs in rings])
    print('Num results: %d' % len(chem))
    print('[QED]       Avg: %s' % fmt(np.mean([c['qed'] for c in chem]), 3))
    print('[SA]        Avg: %s' % fmt(np.mean([c['sa'] for c in chem]), 3))
    print('[logP]      Avg: %s' % fmt(np.mean([c['logp'] for c in chem]), 2))
    print('[5-rule]    %s%%' % fmt(100 * np.mean([int(c['lipinski']) == 5 for c in chem]), 1))
    print('[Rings/mol] Avg: %s' % fmt(n_rings.mean(), 3))
    print('[Acyclic]   %s%%' % fmt(100 * np.mean(n_rings == 0), 1))
    print('[Ring>=10]  %s%%' % fmt(100 * n_large.sum() / n_rings.sum(), 1))

    # out-of-box poses carry penalties of order 1e6 kcal/mol
    vina = np.array([[r['vina']['score_only'][0]['affinity'], r['vina']['dock'][0]['affinity']]
                     for r in results if docked(r)])
    vina = vina[(np.abs(vina) < 1e3).all(axis=1)]
    print('Num docked: %d' % len(vina))
    print('[Vina Score] Avg: %s | Med: %s' % (fmt(vina[:, 0].mean(), 2), fmt(np.median(vina[:, 0]), 2)))
    print('[Vina Dock]  Avg: %s | Med: %s' % (fmt(vina[:, 1].mean(), 2), fmt(np.median(vina[:, 1]), 2)))

## ChIEDiff

In [20]:
if Globals.our_results is not None:
    flat_our = [r for pr in Globals.our_results for r in pr]
    print('Num entries: %d | with a molecule: %d | pockets: %d' % (
        len(flat_our), sum(r.get('mol') is not None for r in flat_our),
        sum(any(docked(r) for r in pr) for pr in Globals.our_results)))
    print_results(flat_our)

Num entries: 9999 | with a molecule: 9857 | pockets: 99
Num results: 8338
[QED]       Avg: 0.475
[SA]        Avg: 0.539
[logP]      Avg: 2.01
[5-rule]    70.2%
[Rings/mol] Avg: 3.385
[Acyclic]   4.3%
[Ring>=10]  2.5%
Num docked: 8339
[Vina Score] Avg: -9.56 | Med: -10.06
[Vina Dock]  Avg: -10.01 | Med: -9.90
